# Week 9 - Regression Model
This week, you will learn to run a Multiple Regression Model.  When you are trying to predict a continuous variable you are building a regression model.  This week you will use the Boston Housing Data set with the B and CHAS columns deleted.<br>

- CRIM - per capita crime rate by town
- ZN - proportion of residential land zoned for lots over 25,000 sq.ft.
- INDUS - proportion of non-retail business acres per town.
- CHAS - Charles River dummy variable (1 if tract bounds river; 0 otherwise)
- NOX - nitric oxides concentration (parts per 10 million)
- RM - average number of rooms per dwelling
- AGE - proportion of owner-occupied units built prior to 1940
- DIS - weighted distances to five Boston employment centres
- RAD - index of accessibility to radial highways
- TAX - full-value property-tax rate per 10,000 dollars
- PTRATIO - pupil-teacher ratio by town
- B - 1000(Bk - 0.63)^2, where Bk is the proportion of blacks by town
- LSTAT - % lower status of the population
- MEDV - Median value of owner-occupied homes in $1000's


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as so


In [2]:
# Read in the dataset
boston = pd.read_csv("boston_clean.csv")
boston.head()

,Unnamed,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0,0.00632,18.0,-2.31,NaN,0.538,NaN,65.2,4.0900,1.0,NaN,15.3,NaN,4.98,24.0
1,1,0.02731,0.0,7.07,Other,NaN,NaN,78.9,4.9671,2.0,242,17.8,NaN,NaN,21.6
2,2,NaN,0.0,7.07,Other,0.469,7.185,61.1,4.9671,2.0,NaN,NaN,NaN,NaN,34.7
3,3,0.03237,0.0,2.18,Other,NaN,6.998,45.8,6.0622,3.0,222',NaN,394.63,NaN,33.4
4,4,0.06905,0.0,2.18,Other,NaN,7.147,NaN,6.0622,NaN,222',NaN,NaN,NaN,36.2


# Drop Unnamed, B, RAD, and CHAS columns

In [3]:
LN = ['Unnamed', 'B', 'RAD', 'CHAS']
boston.drop(columns = LN, axis = 1, inplace = True)

In [4]:
boston.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     456 non-null    float64
 1   ZN       463 non-null    float64
 2   INDUS    506 non-null    float64
 3   NOX      328 non-null    float64
 4   RM       329 non-null    float64
 5   AGE      317 non-null    float64
 6   DIS      339 non-null    float64
 7   TAX      336 non-null    object 
 8   PTRATIO  331 non-null    float64
 9   LSTAT    321 non-null    float64
 10  MEDV     506 non-null    float64
dtypes: float64(10), object(1)
memory usage: 43.6+ KB


In [5]:
# Delete Duplicate Records
boston_dup = boston[boston.duplicated(keep = 'first')]
if boston_dup.size != 0:
    boston.drop_duplicates(keep = 'first', ignore_index = True, inplace = True)
boston.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     456 non-null    float64
 1   ZN       463 non-null    float64
 2   INDUS    506 non-null    float64
 3   NOX      328 non-null    float64
 4   RM       329 non-null    float64
 5   AGE      317 non-null    float64
 6   DIS      339 non-null    float64
 7   TAX      336 non-null    object 
 8   PTRATIO  331 non-null    float64
 9   LSTAT    321 non-null    float64
 10  MEDV     506 non-null    float64
dtypes: float64(10), object(1)
memory usage: 43.6+ KB


In [6]:
boston['TAX'].unique()

array([nan, '242', "222'", '311', '307', '279', '252', '233', '243',
       '469', '313', '256', '284', '337', '345', '305', '398', '281',
       '247', '270', '276', '384', '432', '188', '437', '403', '296',
       '193', '265', '329', '402', '348', '224', '277', '300', '330',
       '315', '264', '223', '254', '216', '198', '285', '241', '293',
       '245', '289', '358', '222', '304', '287', '422', '370', '352',
       '351', '280', '335', '411', '334', '666', '711', '391', '273'],
      dtype=object)

In [7]:
boston.loc[boston['TAX'] == "222'", "TAX"] = '222'
boston['TAX'] = boston['TAX'].astype(float)
boston.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   CRIM     456 non-null    float64
 1   ZN       463 non-null    float64
 2   INDUS    506 non-null    float64
 3   NOX      328 non-null    float64
 4   RM       329 non-null    float64
 5   AGE      317 non-null    float64
 6   DIS      339 non-null    float64
 7   TAX      336 non-null    float64
 8   PTRATIO  331 non-null    float64
 9   LSTAT    321 non-null    float64
 10  MEDV     506 non-null    float64
dtypes: float64(11)
memory usage: 43.6 KB


In [8]:
num_cols = boston.select_dtypes(include =['number']).columns
boston[num_cols].min()

CRIM         0.00632
ZN           0.00000
INDUS      -11.93000
NOX          0.39200
RM           3.56100
AGE          2.90000
DIS         -4.23900
TAX        188.00000
PTRATIO     12.60000
LSTAT        1.73000
MEDV         5.00000
dtype: float64

# Make sure percentage column minimum value is 0 or more
Delete missing INDUS and DIS records with less than 0

In [9]:
b_index_1 = boston[boston['INDUS'] < 0].index
boston.drop(b_index_1, axis = 0, inplace= True)

In [10]:
b_index_2 = boston[boston['DIS'] < 0].index
boston.drop(b_index_2, axis = 0, inplace= True)

In [11]:
boston[num_cols].max()

CRIM        88.9762
ZN         100.0000
INDUS       27.7400
NOX          0.8710
RM           8.7250
AGE        100.0000
DIS         12.1265
TAX        711.0000
PTRATIO     22.0000
LSTAT       37.9700
MEDV        50.0000
dtype: float64

In [12]:
# Number of Missing Values
boston.isnull().sum()

CRIM        48
ZN          42
INDUS        0
NOX        176
RM         174
AGE        188
DIS        164
TAX        167
PTRATIO    173
LSTAT      183
MEDV         0
dtype: int64

# Question 1  - Creating a nominal variable to stratify the data
- Create a new variable 'ZONE_TYPE' using the where 'INDUS' column is greater than or equal 10 is 'Industrial Hub', otherwise, 'Residential/Light'.
- Drop the INDUS column.
- Display the distribution of the 'ZONE_TYPE'.

# Question 2 - Stratify the dataset by ZONE_TYPE column
- Write the train_test_split from sklearn.model_selection import .....
- Create a train and test data set from boston data set where test_size = 0.30, stratify using boston['ZONE_TYPE'], and random_state = 42.

# Question 3 - ZONE_TYPE column distribution in train and test data set
- In a code cell, find the percentage in each category in ZONE_TYPE column in the train.
- In a code cell, find the percentage in each category in ZONE_TYPE column in the test data set


# Exploring the Training Dataset
It is now time to explore the training dataset.<br>

# Question 4 - Describe and find correlations
- Create a list of numeric data called num_cols of the train dataset
- Run the describe().T on the train dataset

# Correlation with the Median Sales price
Correlation strength scale (absolute value)

• 0.90 to 1.00 → Excellent<br>
• 0.75 to 0.90 → Strong<br>
• 0.50 to 0.75 → Moderate<br>
• 0.30 to 0.50 → Weak<br>
• 0.00 to 0.30 → Very weak to none<br>

**Remember:**<br>
• The sign (+ or –) only tells direction, not strength.<br>
• Strength is based on the absolute value.

# Question 5 - Correlated with Target Variable
- Create a correlation matrix of the columns in the num_cols list and store it in the correl_matrix.
- Find the absolute value of the correlation matrix for the 'MEDV' column, and sort it in descending order.

# Question 6 - Data Visualization
For a correlation value greater than 0.50, you will plot the following scatterplots in the train data set.
- Using train.plot.scatter to create a scatterplot of RM and MEDV.  The color (c) is 'DarkBlue', title is "Avg No. of rooms vs Median Sales Price".
- In another code cell, train.plot.scatter to create a scatterplot of LSTAT and MEDV.  The color (c) is 'Purple', title is "% Lower status of the Pop. vs Median Sales Price".
- In another code cell, train.plot.scatter to create a scatterplot of PTRATIO and  MEDV.  The color (c) is "DarkGreen", title is "Pupil-teacher ratio by town vs Median Sales Price".
- In another code cell, train.plot.scatter to create a scatterplot of TAX and  MEDV.  The color (c) is 'Brown', title is "Full-Value Property-tax rate per 10,000 dollars vs Median Sales Price".
- In another code cell, train.plot kind ="hist" , y = 'MEDV', color ('c') is Green (g), edgecolor is black ('k'), title = "Median Sales Price", figsize (5, 5) bins = 30 

# Question 7 - Correlation Matrix
Which feature Values are correlated.  For highly correlated values you will need to remove one of the columns.
- Display the correl_matrix variable
- Assign 'MEDV' from the train data set to the variable y.
- Drop 'MEDV' from the train data set with axis = 1 and without the inplace parameter assign it to the variable X.
- Assign 'MEDV' from the test data set to the variable y_test.
- Drop 'MEDV' from the test data set with axis = 1 and without the inplace parameter assign it to the variable X_test.

# Dealing with Missing Numerical Values
The SimpleImputer function is used to replace the missing values. In the numeric columns we must decide how they will be replaced. We will replace missing values with the median value. <br><br>
A pipeline is a series of steps that the data set goes through the prepare the data to create a model. <br>

# Question 8 - Creating a Pipeline for Numerical data
- from sklearn.impute impute SimpleImputer
- from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
- from sklearn.pipeline import Pipeline
- Create number features from X that are np.number put it in a variable numeric_features
- Create a Pipeline called numeric_transformer 
-        that imputes the missing values with median 
-        and scaler with the StandardScaler
- Create a Pipeline called numeric_transformer_1 
-        that imputes the missing values with median 
-        and scaler with the RobustScaler

# Question 9 - Missing Categorical Values 
Creating pipeline for Categorical columns<br>
- Create a list of all the 'object' called categorical_features
- Create a categorical Pipeline called categorical_transformer 
-         that impute missing values using SimpleImputer strategy equal to "most_frequent" 
-         and onehot OneHotEncoder with handle_unknown= "ignore"

# Question 10 - Create the data preprocessing pipeline
Use the code below to create the data preprocessing pipeline.
- from sklearn.compose import ColumnTransformer
- Create a ColumnTransformer called preprocessor
-         name 'num' the numeric_transformer and called the numeric_features
-         name 'cat', the categorical_transformer, categorical_features


# Model building
Since we are trying to predict total price as a continuous value, we will fit a regression model.  There are several different models.  
- **Multiple Linear Regression**
This week, you will create all the models above using the Boston Housing Data. We will also evaluate the models using cross-validation.


# Question 11  Mutliple Linear Regression - StandardScaler
- from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
- from sklearn.model_selection import KFold, cross_val_score
- from sklearn.linear_model import LinearRegression
- from sklearn.metrics import mean_squared_error, r2_score
- create a make_pipeline(preprocessor called lin_reg, TransformedTargetRegressor(LinearRegression(), transformer is StandardScaler()
- fit the lin_reg with X features and y target
- create y_pred from lin_reg with predict X
- create mean_squared_error with y and y_pred parameters raised to 0.5 called lin_reg_rmse_train
- create y_pred_test as the lin_reg.predict for X_test
- credit the lin_reg_rmse_test as mean_squared_error for y_test, and y_pred_test raised to 0.5
- print "Standard Scaler"
- print "Training ", lin_reg_rmse_train
- print "Testing ", lin_reg_rmse_test

# Model with Robust Scaler
To compare with the Standard Scaler Model used to give equal footing for each column in the model with the following formula.

(X - Mean)/Standard Deviation

There is another option called Robust Scaler.  The formula for the Robust Scaler is as follows.<br>
(X - Median)/IQR

In this homework assignment, you will compare the two models.

# Question 12
- create a make_pipeline(preprocessor_1 called lin_reg1, TransformedTargetRegressor(LinearRegression(), transformer is RobustScaler()
- fit the lin_reg1 with X features and y target
- create y_pred1 from lin_reg with predict X
- create mean_squared_error with y and y_pred parameters raised to 0.5 called lin_reg_rmse_train1
- create y_pred_test1 as the lin_reg1.predict for X_test
- credit the lin_reg_rmse_test1 as mean_squared_error for y_test, and y_pred_test1 raised to 0.5
- print("Robust Scaler")
- print "Training ", lin_reg_rmse_train1
- print "Testing ", lin_reg_rmse_test1